# Amino Acid Statistical Analysis

Python equivalent of the MATLAB `Analysis.m` blog posts.  
Change `AMINO` in the next cell to analyse any of the 20 standard amino acids.

**Requires:** `python/run_pipeline.py` to have been run first (generates `aminoData/` and `contData/`).

| Section | Content |
|---|---|
| 1 | B-factor and bond geometry statistics |
| 2 | Backbone φ/ψ circular statistics |
| 3 | Uniformity tests (Rayleigh, Omnibus, Rao, V-test) |
| 4 | Measures of association (circular-circular, circular-linear) |
| 5 | Ramachandran plot |
| 6 | Side-chain χ statistics (χ₁–χ₄) |
| 7 | Statistical potential maps (−log p) and gradients |

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Change AMINO to analyse a different residue type.
# Valid values: ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL
AMINO = "TYR"
N_ANG = 180   # angular resolution of precomputed maps (must match run_pipeline.py)

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats as scipy_stats
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))

from protein_periodicity.circ_stat import (
    circ_mean, circ_r, circ_var, circ_std, circ_median,
    circ_skewness, circ_kurtosis,
    circ_rtest, circ_otest, circ_raotest, circ_vtest,
    circ_corrcc, circ_corrcl,
    circ_rad2ang, rmatrix,
)
from protein_periodicity.analysis import load_amino_data, load_potential, SINGLE_COLS

AMINO_DIR = Path('../aminoData')
CONT_DIR  = Path('../contData')
IMG_DIR   = Path('../images')
IMG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print(f'Analysing: {AMINO}   (N_ANG={N_ANG})')

---
## Data loading

In [ ]:
csv_path = AMINO_DIR / f'{AMINO}.csv'
if not csv_path.exists():
    raise FileNotFoundError(
        f'No data for {AMINO}.\n'
        f'Run:  cd python/ && python run_pipeline.py'
    )

data = load_amino_data(csv_path)   # shape (n, 12), rows with NaN phi/psi already dropped
n    = len(data)

# Named column views
phi   = data[:, 0]   # radians
psi   = data[:, 1]   # radians
dC_CA = data[:, 2]   # Å
dN_CA = data[:, 3]   # Å
dN_C  = data[:, 4]   # Å
dPpl  = data[:, 5]   # Å  (dP_plane = peptide bond C′–N)
ang   = data[:, 6]   # bond angle N-Cα-C′ (°)
bfac  = data[:, 7]   # B-factor
chi1  = data[:, 8]   # radians (NaN for GLY, ALA)
chi2  = data[:, 9]
chi3  = data[:, 10]
chi4  = data[:, 11]

has = {f'chi{k}': np.isfinite(data[:, 7+k]).any() for k in range(1, 5)}

print(f'\n{"="*50}')
print(f'  Amino acid : {AMINO}')
print(f'  Sample size: {n}')
print(f'  Columns    : {SINGLE_COLS}')
print(f'  Side-chains: ' + '  '.join(f'{k}={"✓" if v else "—"}' for k,v in has.items()))
print(f'{"="*50}')

---
## 1 · B-factor and bond geometry statistics

In [ ]:
def lin_stats(x, label):
    v = x[np.isfinite(x)]
    return {'label': label, 'n': len(v),
            'Mean':               round(float(np.mean(v)),      2),
            'Median':             round(float(np.median(v)),    2),
            'Variance':           round(float(np.var(v,ddof=1)),2),
            'Standard deviation': round(float(np.std(v,ddof=1)),2),
            'Skewness':           round(float(scipy_stats.skew(v)),       2),
            'Kurtosis':           round(float(scipy_stats.kurtosis(v)),   2)}

# B-factor table
bf = lin_stats(bfac, 'B-factor')
rows = ['Mean','Median','Variance','Standard deviation','Skewness','Kurtosis']
print(f'SAMPLE SIZE: {n}\n')
df_bf = pd.DataFrame({'': rows, 'B-factor': [bf[r] for r in rows]})
display(df_bf.set_index(''))

In [ ]:
# Bond geometry table + correlation matrix
cc = lin_stats(dC_CA, 'C–Cα')
nc = lin_stats(dN_CA, 'N–Cα')
pp = lin_stats(dPpl,  'Peptide bond')

mask = np.isfinite(dC_CA) & np.isfinite(dN_CA) & np.isfinite(dPpl)
C = np.corrcoef(np.stack([dC_CA[mask], dN_CA[mask], dPpl[mask]]))

rows_g = rows + ['corrcoef [C-CA]', 'corrcoef [N-CA]', 'corrcoef [pept]']
def geo_col(s, ci):
    return [s[r] for r in rows] + [round(C[ci,0],2), round(C[ci,1],2), round(C[ci,2],2)]

df_geo = pd.DataFrame({
    '':               rows_g,
    'C–Cα (dC_CA)':  geo_col(cc, 0),
    'N–Cα (dN_CA)':  geo_col(nc, 1),
    'Peptide (dPpl)': geo_col(pp, 2),
})
display(df_geo.set_index(''))

---
## 2 · Backbone φ/ψ circular statistics

In [ ]:
def circ_stats_row(alpha, name):
    v = alpha[np.isfinite(alpha)]
    mu     = circ_mean(v)
    med    = circ_median(v[:min(len(v), 800)])   # O(n²) — cap for speed
    R      = circ_r(v)
    S      = circ_var(v)
    s, s0  = circ_std(v)
    b      = circ_skewness(v)
    k      = circ_kurtosis(v)
    return {
        'name':                name,
        'n':                   len(v),
        'Mean (°)':            round(np.degrees(mu),  2),
        'Median (°)':          round(np.degrees(med), 2),
        'R Length':            round(R,  2),
        'Variance':            round(S,  2),
        'Standard deviation':  round(s,  2),
        'Standard deviation 0':round(s0, 2),
        'Skewness':            round(b,  2),
        'Kurtosis':            round(k,  2),
    }

phi_s = circ_stats_row(phi, 'φ (PHI)')
psi_s = circ_stats_row(psi, 'ψ (PSI)')

stat_keys = ['Mean (°)','Median (°)','R Length','Variance',
             'Standard deviation','Standard deviation 0','Skewness','Kurtosis']
df_pp = pd.DataFrame({
    '':        stat_keys,
    'PHI':     [phi_s[k] for k in stat_keys],
    'PSI':     [psi_s[k] for k in stat_keys],
})
display(df_pp.set_index(''))

---
## 3 · Inferential statistics for φ-ψ

### Tests for Uniformity

In [ ]:
def uniformity_block(alpha, name):
    v = alpha[np.isfinite(alpha)]
    p_r, _       = circ_rtest(v)
    p_o, _       = circ_otest(v)
    p_rao, _, _  = circ_raotest(v)
    p_v, _       = circ_vtest(v, 0.0)
    print(f'  {name}:')
    print(f'    Rayleigh Test,    P = {p_r:.2f}')
    print(f'    Omnibus Test,     P = {p_o:.2f}')
    print(f'    Rao Spacing Test, P ≥ {p_rao:.2f}')
    print(f'    V Test (r=0),     P = {p_v:.2f}')
    return {'Rayleigh': p_r, 'Omnibus': p_o, 'Rao': p_rao, 'V-test': p_v}

print(f'Tests for Uniformity — {AMINO}\n')
phi_tests = uniformity_block(phi, 'φ (PHI)')
psi_tests = uniformity_block(psi, 'ψ (PSI)')

---
## 4 · Measures of association φ-ψ

In [ ]:
print(f'Measures of Association — {AMINO}\n')
print('Circular–Circular Association')

rho, p = circ_corrcc(phi, psi)
print(f'  Circ-circ corr φ–ψ:          coeff={rho:+.2f}  pval={p:.3f}')

print('\nCircular–Linear Association')
for angle, alpha in [('φ', phi), ('ψ', psi)]:
    for lname, lvec in [('dC_CA', dC_CA), ('dN_CA', dN_CA)]:
        mask = np.isfinite(alpha) & np.isfinite(lvec)
        rho, p = circ_corrcl(alpha[mask], lvec[mask])
        print(f'  Circ-lin corr {angle}–{lname}:   coeff={rho:.2f}  pval={p:.3f}')

---
## 5 · Ramachandran plot

In [ ]:
phi_d = np.degrees(phi)
psi_d = np.degrees(psi)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
ax = axes[0]
ax.scatter(phi_d, psi_d, s=6, alpha=0.45, color='steelblue', lw=0)
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
ax.axhline(0, c='k', lw=0.4); ax.axvline(0, c='k', lw=0.4)
ax.set_xlabel('φ (°)'); ax.set_ylabel('ψ (°)')
ax.set_title(f'{AMINO} — Ramachandran  (n={n})')
for lab, px, py in [('α', -65, -45), ('β', -120, 125), ('Lα', 58, 42)]:
    ax.annotate(lab, xy=(px, py), fontsize=13, fontweight='bold', color='navy')

# Hexbin density
ax = axes[1]
hb = ax.hexbin(phi_d, psi_d, gridsize=24, cmap='YlOrRd', extent=[-180,180,-180,180])
plt.colorbar(hb, ax=ax, label='count')
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
ax.axhline(0, c='k', lw=0.4); ax.axvline(0, c='k', lw=0.4)
ax.set_xlabel('φ (°)'); ax.set_ylabel('ψ (°)')
ax.set_title(f'{AMINO} — Ramachandran (density)')

plt.tight_layout()
plt.savefig(IMG_DIR / f'{AMINO}_Rama_phipsi.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved → images/{AMINO}_Rama_phipsi.png')

---
## 6 · Side-chain χ statistics

In [ ]:
def chi_section(chi_arr, chi_name, phi, psi, prev_chis):
    """Print statistics, uniformity tests, and correlations for one chi angle."""
    v = chi_arr[np.isfinite(chi_arr)]
    if len(v) < 5:
        print(f'  {chi_name}: insufficient data (n={len(v)})')
        return

    s = circ_stats_row(chi_arr, chi_name)
    print(f'\n### Statistics {chi_name}  (n={s["n"]})')
    df = pd.DataFrame({'': stat_keys, chi_name: [s[k] for k in stat_keys]})
    display(df.set_index(''))

    print(f'\n### Inferential Statistics {chi_name}\nTests for Uniformity:')
    uniformity_block(chi_arr, chi_name)

    print('\nCorrelations:')
    for aname, alpha in [('φ', phi), ('ψ', psi)] + prev_chis:
        mask = np.isfinite(alpha) & np.isfinite(chi_arr)
        if mask.sum() < 5:
            continue
        rho, p = circ_corrcc(alpha[mask], chi_arr[mask])
        print(f'  Circ-circ corr {aname}–{chi_name}:  coeff={rho:+.2f}  pval={p:.3f}')

chi_data = [
    (chi1, 'χ₁', []),
    (chi2, 'χ₂', [('χ₁', chi1)]),
    (chi3, 'χ₃', [('χ₁', chi1), ('χ₂', chi2)]),
    (chi4, 'χ₄', [('χ₁', chi1), ('χ₂', chi2), ('χ₃', chi3)]),
]

any_chi = False
for chi_arr, chi_name, prev in chi_data:
    if np.isfinite(chi_arr).sum() >= 5:
        any_chi = True
        chi_section(chi_arr, chi_name, phi, psi, prev)

if not any_chi:
    print(f'{AMINO} has no side-chain dihedral angles (e.g. GLY or ALA).')

In [ ]:
# χ₁ histogram (if available)
if np.isfinite(chi1).sum() >= 5:
    chi1_d = np.degrees(chi1[np.isfinite(chi1)])
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(chi1_d, bins=36, range=(-180, 180),
            color='steelblue', edgecolor='white', linewidth=0.3)
    ax.set_xlabel('χ₁ (°)'); ax.set_ylabel('count')
    ax.set_title(f'{AMINO} — χ₁ distribution  (n={len(chi1_d)})')
    ax.set_xlim(-180, 180)
    plt.tight_layout()
    plt.savefig(IMG_DIR / f'{AMINO}_chi1_hist.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → images/{AMINO}_chi1_hist.png')
else:
    print(f'{AMINO}: no χ₁ data — histogram skipped.')

---
## 7 · Statistical potential maps  (−log p) and gradients

In [ ]:
result = load_potential(CONT_DIR, AMINO, n_ang=N_ANG)

if result is None:
    print(f'No precomputed potentials for {AMINO}.\n'
          f'Run:  cd python/ && python run_pipeline.py')
else:
    pots = result['potentials']
    print(f'{AMINO}: {len(pots)} potential maps  (n_total={result["n_total"]}, n_ang={result["n_ang"]})')
    for p in pots:
        print(f'  {"–".join(p["pair"]):12s}  n={p["n"]:5d}  '
              f'κ=[{p["kappa1"]:.2f},{p["kappa2"]:.2f}]  '
              f'E∈[{p["E"].min():.2f},{p["E"].max():.2f}]')

In [ ]:
if result is not None:
    pots = result['potentials']
    n_pairs = len(pots)
    grid_deg = np.linspace(-180, 180, N_ANG, endpoint=False)
    ext = [-180, 180, -180, 180]

    fig, axes = plt.subplots(n_pairs, 2, figsize=(13, 4.5 * n_pairs), squeeze=False)

    for row, pot in enumerate(pots):
        a1, a2 = pot['pair']
        label  = f'{a1}–{a2}'
        E  = pot['E']
        Gx = pot['Gx']
        Gy = pot['Gy']
        Gnorm = np.sqrt(Gx**2 + Gy**2)

        # −log p map
        ax = axes[row, 0]
        E_clip = np.clip(E, 0, np.percentile(E, 98))
        im = ax.imshow(rmatrix(E_clip), extent=ext, origin='lower',
                       cmap='RdYlGn_r', aspect='auto')
        plt.colorbar(im, ax=ax, label='−log p')
        ax.axhline(0, c='k', lw=0.4, ls='--')
        ax.axvline(0, c='k', lw=0.4, ls='--')
        ax.set_xlabel(f'{a2} (°)'); ax.set_ylabel(f'{a1} (°)')
        ax.set_title(f'{AMINO} — {label}  (n={pot["n"]})')

        # Gradient norm
        ax = axes[row, 1]
        G_clip = np.clip(Gnorm, 0, np.percentile(Gnorm, 98))
        im2 = ax.imshow(rmatrix(G_clip), extent=ext, origin='lower',
                        cmap='hot_r', aspect='auto')
        plt.colorbar(im2, ax=ax, label='|∇(−log p)|')
        ax.axhline(0, c='k', lw=0.4, ls='--')
        ax.axvline(0, c='k', lw=0.4, ls='--')
        ax.set_xlabel(f'{a2} (°)'); ax.set_ylabel(f'{a1} (°)')
        ax.set_title(f'{AMINO} — ‖∇‖ {label}')

    plt.suptitle(f'{AMINO} — Statistical potentials  (N_ANG={N_ANG})', fontsize=14, y=1.01)
    plt.tight_layout()
    out = IMG_DIR / f'{AMINO}_potentials.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out}')

---
## Summary

| Section | Result |
|---------|--------|
| Sample size | see cell 4 |
| B-factor & bond geometry | linear statistics + correlation matrix |
| φ/ψ circular statistics | mean, R, variance, std, skewness, kurtosis |
| Uniformity tests | Rayleigh, Omnibus, Rao, V-test |
| Associations | φ–ψ circ-circ; φ/ψ–dC_CA/dN_CA circ-linear |
| Ramachandran | scatter + hexbin density |
| χ₁–χ₄ | stats + tests + correlations (if available) |
| Potential maps | −log p + gradient norm for all dihedral pairs |